## **05 MongoDB Python Aggregation**

### 0. 기본 pymongo 템플릿 코드
> sample_mflix 데이터셋을 기반으로, 지금까지 익힌 MongoDB aggregation 문법을 pymongo 에서 어떻게 적용해서 사용할 수 있는지를 알아보기로 함

In [24]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/") # MongoDB 연결
db = client.sample_mflix # 데이터베이스 선택
movies = db.movies       # 컬렉션 선택

In [4]:
movie_list = list(movies.find())
len(movie_list)

23539

In [10]:
movie1 = movie_list[0] # 첫번째 영화 문서 출력
type(movie1), movie1 # 딕셔너리 (document: JSON)

(dict,
 {'_id': ObjectId('573a1390f29313caabcd4135'),
  'plot': 'Three men hammer on an anvil and pass a bottle of beer around.',
  'genres': ['Short'],
  'runtime': 1,
  'cast': ['Charles Kayser', 'John Ott'],
  'num_mflix_comments': 1,
  'title': 'Blacksmith Scene',
  'fullplot': 'A stationary camera looks at a large anvil with a blacksmith behind it and one on either side. The smith in the middle draws a heated metal rod from the fire, places it on the anvil, and all three begin a rhythmic hammering. After several blows, the metal goes back in the fire. One smith pulls out a bottle of beer, and they each take a swig. Then, out comes the glowing metal and the hammering resumes.',
  'countries': ['USA'],
  'released': datetime.datetime(1893, 5, 9, 0, 0),
  'directors': ['William K.L. Dickson'],
  'rated': 'UNRATED',
  'awards': {'wins': 1, 'nominations': 0, 'text': '1 win.'},
  'lastupdated': '2015-08-26 00:03:50.133000000',
  'year': 1893,
  'imdb': {'rating': 6.2, 'votes': 1189, 'id

In [11]:
movie1.keys() # 영화 문서의 필드 확인

dict_keys(['_id', 'plot', 'genres', 'runtime', 'cast', 'num_mflix_comments', 'title', 'fullplot', 'countries', 'released', 'directors', 'rated', 'awards', 'lastupdated', 'year', 'imdb', 'type', 'tomatoes'])

### 다양한 aggregate() 문법 적용
- MongoDB aggregation 문법은 find() 가 아닌, aggregate() 메서드를 사용해야 함

**1. $match: 이 스테이지는 쿼리와 유사한 방식으로 문서를 필터링합니다.**

> 결과가 너무 많기 때문에, $limit 문법도 함께 사용하기로 함

In [16]:
# "Aciont" 장르의 영화 한 편 조회
pipeline = [
    {"$match": {"genres" : "Action"}}, # genres 필드에 "Action"이 포함된 영화 찾기
      {"$limit":1}                     # 데이터는 하나만 조회
]

for movie in movies.aggregate(pipeline):
    print(movie)

{'_id': ObjectId('573a1390f29313caabcd5293'), 'plot': "Young Pauline is left a lot of money when her wealthy uncle dies. However, her uncle's secretary has been named as her guardian until she marries, at which time she will officially take ...", 'genres': ['Action'], 'runtime': 199, 'cast': ['Pearl White', 'Crane Wilbur', 'Paul Panzer', 'Edward Josè'], 'num_mflix_comments': 1, 'poster': 'https://m.media-amazon.com/images/M/MV5BMzgxODk1Mzk2Ml5BMl5BanBnXkFtZTgwMDg0NzkwMjE@._V1_SY1000_SX677_AL_.jpg', 'title': 'The Perils of Pauline', 'fullplot': 'Young Pauline is left a lot of money when her wealthy uncle dies. However, her uncle\'s secretary has been named as her guardian until she marries, at which time she will officially take possession of her inheritance. Meanwhile, her "guardian" and his confederates constantly come up with schemes to get rid of Pauline so that he can get his hands on the money himself.', 'languages': ['English'], 'released': datetime.datetime(1914, 3, 23, 0, 0), '

**2. $group: 이 스테이지는 특정 필드를 기준으로 문서를 그룹화하고, 각 그룹에 대해 다양한 연산을 수행할 수 있습니다.**

In [17]:
## 감독별 영화 수 세기
pipeline = [
    {"$group":{"_id": "$directors", "count":{"$sum":1}}}, # directors 필드를 기준으로 그룹화하여 영화 수 세기
    {"$sort" : {"count": -1}}, # 영화 수 기준으로 내림차순
    {"$limit": 10} # 상위 10명 감독만 조회
]

for group in movies.aggregate(pipeline):
    print(group)

{'_id': None, 'count': 265}
{'_id': ['Woody Allen'], 'count': 39}
{'_id': ['Takashi Miike'], 'count': 33}
{'_id': ['Werner Herzog'], 'count': 31}
{'_id': ['Alfred Hitchcock'], 'count': 31}
{'_id': ['John Huston'], 'count': 30}
{'_id': ['Martin Scorsese'], 'count': 29}
{'_id': ['Sidney Lumet'], 'count': 29}
{'_id': ['John Ford'], 'count': 29}
{'_id': ['Steven Spielberg'], 'count': 28}


**3. $sort: 이 스테이지는 특정 필드를 기준으로 문서를 정렬합니다.**

In [ ]:
# title 기준 오름차순으로 상위 5개 영화 조회 (abc 순으로 정렬)
pipeline =[
    {"$sort":{"title": 1}},                 # title 필드를 기준으로 내림차순 정렬
    {"$project":{"_id": 0, "title": 1}},
    {"$limit": 5}   
]

**4. $limit: 이 스테이지는 출력되는 문서의 수를 제한합니다.**

In [18]:
pipeline = [
    {"$limit": 1}
]
for movie in movies.aggregate(pipeline):
    print(movie)
    
# db.movies.aggregate( [ { $limit: 1 } ] )    

{'_id': ObjectId('573a1390f29313caabcd4135'), 'plot': 'Three men hammer on an anvil and pass a bottle of beer around.', 'genres': ['Short'], 'runtime': 1, 'cast': ['Charles Kayser', 'John Ott'], 'num_mflix_comments': 1, 'title': 'Blacksmith Scene', 'fullplot': 'A stationary camera looks at a large anvil with a blacksmith behind it and one on either side. The smith in the middle draws a heated metal rod from the fire, places it on the anvil, and all three begin a rhythmic hammering. After several blows, the metal goes back in the fire. One smith pulls out a bottle of beer, and they each take a swig. Then, out comes the glowing metal and the hammering resumes.', 'countries': ['USA'], 'released': datetime.datetime(1893, 5, 9, 0, 0), 'directors': ['William K.L. Dickson'], 'rated': 'UNRATED', 'awards': {'wins': 1, 'nominations': 0, 'text': '1 win.'}, 'lastupdated': '2015-08-26 00:03:50.133000000', 'year': 1893, 'imdb': {'rating': 6.2, 'votes': 1189, 'id': 5}, 'type': 'movie', 'tomatoes': {'

**5. $project: 이 스테이지는 출력되는 문서의 필드를 추가, 제거, 또는 새로 생성합니다.**

In [19]:
pipeline = [
    {"$project": {"_id": 0, "title": 1, "genres": 1}}, {"$limit": 1}
]
for movie in movies.aggregate(pipeline):
    print(movie)
    
# db.movies.aggregate( [ { $project: { _id: 0, title: 1, genres: 1 } }, 
#                        { $limit: 1 } ] )

{'genres': ['Short'], 'title': 'Blacksmith Scene'}


**6. $unwind: 이 스테이지는 배열 필드를 풀어서 각 원소를 별도의 문서로 만듭니다.**

In [20]:
pipeline = [
    {"$unwind": "$genres"}, {"$limit": 6}
]
for movie in movies.aggregate(pipeline):
    print(movie['title'], movie['genres'], movie, sep=' || ')
# db.movies.aggregate( [ { $unwind: "$genres" }, { $limit: 3 } ] )   

Blacksmith Scene || Short || {'_id': ObjectId('573a1390f29313caabcd4135'), 'plot': 'Three men hammer on an anvil and pass a bottle of beer around.', 'genres': 'Short', 'runtime': 1, 'cast': ['Charles Kayser', 'John Ott'], 'num_mflix_comments': 1, 'title': 'Blacksmith Scene', 'fullplot': 'A stationary camera looks at a large anvil with a blacksmith behind it and one on either side. The smith in the middle draws a heated metal rod from the fire, places it on the anvil, and all three begin a rhythmic hammering. After several blows, the metal goes back in the fire. One smith pulls out a bottle of beer, and they each take a swig. Then, out comes the glowing metal and the hammering resumes.', 'countries': ['USA'], 'released': datetime.datetime(1893, 5, 9, 0, 0), 'directors': ['William K.L. Dickson'], 'rated': 'UNRATED', 'awards': {'wins': 1, 'nominations': 0, 'text': '1 win.'}, 'lastupdated': '2015-08-26 00:03:50.133000000', 'year': 1893, 'imdb': {'rating': 6.2, 'votes': 1189, 'id': 5}, 'typ

**7. `$group`과 `$sum`: 이 예제에서는 감독별로 영화를 그룹화하고, 각 그룹의 영화 수를 계산합니다.**

In [21]:
# 'Action' 장르의 영화 5편 출력
pipeline = [
    {"$group": {"_id": "$directors", 
                "count": {"$sum": 1}}}, 
    { "$limit": 5 }
]
for group in movies.aggregate(pipeline):
    print(group)
    
# db.movies.aggregate( [ { $group: { _id: "$directors", count: { $sum: 1 } } }, 
#                                  { $limit: 5 } ] )    

{'_id': ['Alain Resnais'], 'count': 18}
{'_id': ['Gaby Dellal'], 'count': 1}
{'_id': ['John Slattery'], 'count': 1}
{'_id': ['Stèphane Aubier', 'Vincent Patar'], 'count': 1}
{'_id': ['Hugo Fregonese'], 'count': 1}


**8. `$group`과 `$avg`: 이 예제에서는 감독별로 영화를 그룹화하고, 각 그룹의 영화 평점 평균을 계산합니다.**

In [40]:
# 감독별 평균 IMDb 평점 계산
pipeline = [
    {"$group": {"_id": "$directors", "average_rating": {"$avg": "$imdb.rating"}}},
    {"$sort":{"avarage_rating": -1}}, 
    { "$limit": 5 }
]
for group in movies.aggregate(pipeline):
    print(group)
# db.movies.aggregate( [ { $group: { _id: "$directors", average_rating: { $avg: "$imdb.rating" } } }, 
#                                  { $limit: 5 } ] )    

{'_id': ['Rip Torn'], 'average_rating': 4.4}
{'_id': ['Ernie Barbarash'], 'average_rating': 5.5}
{'_id': ['Ted Demme'], 'average_rating': 7.0}
{'_id': ['Sergio Castellitto'], 'average_rating': 6.766666666666667}
{'_id': ['Sami Saif'], 'average_rating': 5.9}


<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 1: 컬렉션에 있는 영화의 수를 계산하세요.</font><br>
</div>

In [41]:
total_movies = movies.count_documents({})
print(total_movies)

23539


<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 2: 평균 영화 길이를 찾으세요.</font><br>
</div>

In [ ]:
pipeline = [
    # runtime 필드가 존재하고, null이 아닌것만 필터링
    { "$match":{"runtime": {"$exists": True, "$ne": None}}},
    # 전체 문서를 대상으로 runtime의 평균($avg) 계산
    {"$group":{
     "_id": None,
     "average_runtime" : {"$avg": "$runtime"}
        }
    }
]

result = list(movies.aggregate(pipeline))
print(result[0]['average_runtime'])

103.78932097696172


<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 3: 각 장르에 대한 영화 수를 계산하세요.</font><br>
</div>

In [56]:
pipeline=[
    {"$unwind":"$genres"},
    {"$group" : {
        "_id": "$genres",
        "count" : {"$sum": 1}
    }},
    {
        "$sort" : {"count":-1}
    }
]

result = list(movies.aggregate(pipeline))
for res in result:
    print(res['_id'], res['count'])

Drama 13789
Comedy 7024
Romance 3665
Crime 2678
Thriller 2658
Action 2539
Documentary 2129
Adventure 2045
Horror 1703
Biography 1404
Family 1311
Mystery 1259
Fantasy 1153
Sci-Fi 1034
History 999
Animation 971
Music 840
War 794
Musical 487
Short 478
Sport 390
Western 274
Film-Noir 105
News 51
Talk-Show 1


<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 4: 2014년 이후에 개봉한 영화를 제목으로 정렬하여 나열하세요.</font><br>
</div>

<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 5: 가장 많은 영화를 제작하는 상위 5개 국가를 찾으세요.</font><br>
</div>

<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 6: 2000년 이후 영화의 연도별 평균 IMDB 평점을 찾으세요.</font><br>
</div>

<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 7:  'Star'라는 단어가 포함된 영화의 제목을 가져오세요.</font><br>
</div>

<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 8:  데이터셋에서 사용 가능한 모든 고유 언어를 나열하세요.</font><br>
</div>

<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 9:  각각의 감독이 제작한 영화 수가 25개 이상인 감독들을 찾으세요.</font><br>
</div>

<div class="alert alert-block" style="border: 2px solid #1565C0;background-color:#E3F2FD;padding:10px">
<font size="3em" style="color:#0D47A1;">연습문제 10:  관람객 평점을 기준으로 상위 5개의 영화를 찾으세요 (1000표 이상의 영화에 한함).</font><br>
</div>